# Day 12 — Train and Evaluate LSTM / GRU Regression Models

**Project:** IPL Player Performance Analytics and Prediction Using Machine Learning and Deep Learning

## Objectives
- Load the sequence tensors produced on Day 11.
- Fit missing-value imputation and feature scaling using the training partition only.
- Train LSTM and GRU regressors for the three IPL next-match targets.
- Use validation MAE for model selection, with validation RMSE as a tie-breaker.
- Evaluate the selected architecture on the untouched chronological test partition.
- Save model artifacts, training curves, metrics, predictions, and a run manifest.

**Important:** this notebook is intended for TensorFlow/Keras. It checks for the dependency before starting and does not fabricate results if the library or input files are missing.

In [6]:
from pathlib import Path
import json
import platform
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    TENSORFLOW_AVAILABLE = True
    TENSORFLOW_IMPORT_ERROR = ""
except Exception as exc:
    TENSORFLOW_AVAILABLE = False
    TENSORFLOW_IMPORT_ERROR = repr(exc)

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
np.random.seed(SEED)
if TENSORFLOW_AVAILABLE:
    tf.random.set_seed(SEED)

current_dir = Path.cwd().resolve()
candidates = [current_dir, *current_dir.parents]
def is_project_root(p):
    return (
        (p / "data" / "processed" / "day11_sequences").is_dir()
        or ((p / "data" / "processed").is_dir() and (p / "notebooks").is_dir())
    )
PROJECT_ROOT = next((p for p in candidates if is_project_root(p)), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path(r"D:\Git\Personal\ipl-player-performance")

SEQUENCE_DIR = PROJECT_ROOT / "data" / "processed" / "day11_sequences"
REPORT_DIR = PROJECT_ROOT / "reports" / "day12"
MODEL_DIR = PROJECT_ROOT / "models" / "day12"
DOCS_DIR = PROJECT_ROOT / "docs"
for folder in [REPORT_DIR, MODEL_DIR, DOCS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("TensorFlow available:", TENSORFLOW_AVAILABLE)
if not TENSORFLOW_AVAILABLE:
    print("TensorFlow import issue:", TENSORFLOW_IMPORT_ERROR)

Project root: D:\Git\Personal\ipl-player-performance
TensorFlow available: True


## 1. Load Day 11 sequence tensors and metadata

This notebook uses the Day 11 feature manifest to preserve feature order. It recreates chronological 70% / 15% / 15% partitions by target match date. Preprocessing is fitted on training samples only.

In [7]:
TASKS = [
    ("batting", "next_match_runs"),
    ("bowling", "next_match_wickets"),
    ("bowling", "next_match_runs_conceded"),
]

def safe_target(target):
    return target.replace("/", "_")

def load_task(dataset, target):
    prefix = f"{dataset}_{safe_target(target)}"
    tensor_path = SEQUENCE_DIR / f"{prefix}_sequences.npz"
    metadata_path = SEQUENCE_DIR / f"{prefix}_metadata.csv"
    feature_path = SEQUENCE_DIR / f"{prefix}_features.json"
    for path in [tensor_path, metadata_path, feature_path]:
        if not path.is_file():
            raise FileNotFoundError(f"Missing Day 11 output: {path}. Run Day 11 first.")
    with np.load(tensor_path) as archive:
        X = archive["X"].astype(np.float32)
        y = archive["y"].astype(np.float32)
    meta = pd.read_csv(metadata_path)
    meta["target_match_date"] = pd.to_datetime(meta["target_match_date"], errors="coerce")
    feature_info = json.loads(feature_path.read_text(encoding="utf-8"))
    features = feature_info["feature_columns_in_order"]
    if X.ndim != 3 or X.shape[1] != 5:
        raise ValueError(f"{prefix}: expected X shape (samples, 5, features); got {X.shape}")
    if len(X) != len(y) or len(y) != len(meta):
        raise ValueError(f"{prefix}: tensor, target, and metadata lengths differ.")
    if X.shape[2] != len(features):
        raise ValueError(f"{prefix}: feature count does not match Day 11 manifest.")
    if meta["target_match_date"].isna().any():
        raise ValueError(f"{prefix}: invalid target match dates in metadata.")
    return X, y, meta, features

def chronological_split(X, y, meta):
    order = np.argsort(meta["target_match_date"].to_numpy(), kind="stable")
    X, y, meta = X[order], y[order], meta.iloc[order].reset_index(drop=True)
    n = len(y)
    if n < 30:
        raise ValueError(f"Only {n} samples; at least 30 are required for 70/15/15 splits.")
    train_end, val_end = int(n * 0.70), int(n * 0.85)
    if train_end < 2 or val_end <= train_end or val_end >= n:
        raise ValueError("Unable to create non-empty chronological partitions.")
    split = {
        "train": (X[:train_end], y[:train_end], meta.iloc[:train_end].copy()),
        "validation": (X[train_end:val_end], y[train_end:val_end], meta.iloc[train_end:val_end].copy()),
        "test": (X[val_end:], y[val_end:], meta.iloc[val_end:].copy()),
    }
    return split

loaded = {}
load_status = []
for dataset, target in TASKS:
    try:
        loaded[(dataset, target)] = load_task(dataset, target)
        X, y, meta, features = loaded[(dataset, target)]
        load_status.append({
            "dataset": dataset, "target": target, "status": "LOADED",
            "samples": len(y), "sequence_length": X.shape[1],
            "feature_count": len(features), "detail": ""
        })
    except Exception as exc:
        load_status.append({
            "dataset": dataset, "target": target, "status": "FAILED",
            "samples": 0, "sequence_length": None, "feature_count": 0,
            "detail": repr(exc)
        })
load_status_df = pd.DataFrame(load_status)
display(load_status_df)

,dataset,target,status,samples,sequence_length,feature_count,detail
0,batting,next_match_runs,LOADED,15821,5,15,
1,bowling,next_match_wickets,LOADED,12335,5,15,
2,bowling,next_match_runs_conceded,LOADED,12335,5,15,


## 2. Define LSTM and GRU architectures

Both models return one continuous value, so these are regression networks. The same training/validation/test partitions and preprocessing rules are used for a fair comparison. Early stopping monitors validation loss.

In [8]:
def build_model(architecture, sequence_length, feature_count):
    model = keras.Sequential(name=f"{architecture.lower()}_regressor")
    model.add(keras.Input(shape=(sequence_length, feature_count)))
    if architecture == "LSTM":
        model.add(layers.LSTM(48, return_sequences=False))
    elif architecture == "GRU":
        model.add(layers.GRU(48, return_sequences=False))
    else:
        raise ValueError(f"Unknown architecture: {architecture}")
    model.add(layers.Dropout(0.20))
    model.add(layers.Dense(24, activation="relu"))
    model.add(layers.Dense(1, activation="linear"))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="mean_squared_error",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")]
    )
    return model

def regression_metrics(actual, predicted):
    return {
        "MAE": float(mean_absolute_error(actual, predicted)),
        "RMSE": float(np.sqrt(mean_squared_error(actual, predicted))),
        "R2": float(r2_score(actual, predicted)) if len(actual) > 1 else np.nan,
        "mean_residual_actual_minus_predicted": float(np.mean(np.asarray(actual) - np.asarray(predicted)))
    }

def transform_sequences(X, imputer=None, scaler=None, fit=False):
    n, seq_len, n_features = X.shape
    flat = X.reshape(-1, n_features)
    if fit:
        imputer = SimpleImputer(strategy="median")
        flat = imputer.fit_transform(flat)
        scaler = StandardScaler()
        flat = scaler.fit_transform(flat)
    else:
        flat = imputer.transform(flat)
        flat = scaler.transform(flat)
    return flat.reshape(n, seq_len, n_features), imputer, scaler

## 3. Train both architectures for each target

Preprocessing is fitted only on the training tensor. The validation set controls early stopping and architecture selection. The test partition is not accessed until both architectures have completed training and the validation-based selection is fixed.

In [9]:
if not TENSORFLOW_AVAILABLE:
    raise ImportError(
        "TensorFlow is required for Day 12. In the ipl-player-performance Conda environment, "
        "install a compatible TensorFlow release, restart the kernel, and rerun this notebook. "
        f"Original import error: {TENSORFLOW_IMPORT_ERROR}"
    )

all_metrics = []
prediction_frames = []
history_frames = []
selection_rows = []
split_rows = []
artifact_rows = []
task_status_rows = []

for dataset, target in TASKS:
    try:
        if (dataset, target) not in loaded:
            raise RuntimeError("Day 11 inputs did not load; see load status table.")
        X_all, y_all, meta_all, feature_names = loaded[(dataset, target)]
        split = chronological_split(X_all, y_all, meta_all)
        X_train_raw, y_train, meta_train = split["train"]
        X_val_raw, y_val, meta_val = split["validation"]
        X_test_raw, y_test, meta_test = split["test"]

        X_train, imputer, scaler = transform_sequences(X_train_raw, fit=True)
        X_val, _, _ = transform_sequences(X_val_raw, imputer=imputer, scaler=scaler)
        # Hold test data aside until the validation-based architecture decision is fixed.
        target_results = []
        trained = {}

        for architecture in ["LSTM", "GRU"]:
            tf.keras.backend.clear_session()
            tf.random.set_seed(SEED)
            model = build_model(architecture, X_train.shape[1], X_train.shape[2])
            callbacks = [
                keras.callbacks.EarlyStopping(
                    monitor="val_loss", patience=12, restore_best_weights=True
                ),
                keras.callbacks.ReduceLROnPlateau(
                    monitor="val_loss", factor=0.5, patience=5, min_lr=1e-5
                ),
            ]
            history = model.fit(
                X_train, y_train,
                validation_data=(X_val, y_val),
                epochs=100,
                batch_size=32,
                callbacks=callbacks,
                verbose=0,
                shuffle=False
            )
            val_pred = model.predict(X_val, verbose=0).reshape(-1)
            val_metrics = regression_metrics(y_val, val_pred)
            target_results.append({
                "dataset": dataset, "target": target, "architecture": architecture,
                "epochs_run": len(history.history.get("loss", [])),
                "validation_MAE": val_metrics["MAE"],
                "validation_RMSE": val_metrics["RMSE"],
                "validation_R2": val_metrics["R2"]
            })
            trained[architecture] = {
                "model": model, "history": history,
                "validation_prediction": val_pred,
                "validation_metrics": val_metrics
            }
            for epoch_index, values in enumerate(history.history.get("loss", []), start=1):
                history_frames.append({
                    "dataset": dataset, "target": target,
                    "architecture": architecture, "epoch": epoch_index,
                    "loss": values,
                    "mae": history.history.get("mae", [np.nan] * len(history.history["loss"]))[epoch_index - 1],
                    "val_loss": history.history.get("val_loss", [np.nan] * len(history.history["loss"]))[epoch_index - 1],
                    "val_mae": history.history.get("val_mae", [np.nan] * len(history.history["loss"]))[epoch_index - 1]
                })

        target_results_df = pd.DataFrame(target_results).sort_values(
            ["validation_MAE", "validation_RMSE"], ascending=True
        )
        selected_architecture = str(target_results_df.iloc[0]["architecture"])
        selection_rows.extend(target_results)
        selected = trained[selected_architecture]

        # Only now use test partition for final reporting.
        X_test, _, _ = transform_sequences(X_test_raw, imputer=imputer, scaler=scaler)
        test_pred = selected["model"].predict(X_test, verbose=0).reshape(-1)
        test_metrics = regression_metrics(y_test, test_pred)
        all_metrics.append({
            "dataset": dataset, "target": target,
            "selected_architecture": selected_architecture,
            "train_samples": len(y_train), "validation_samples": len(y_val),
            "test_samples": len(y_test), "feature_count": len(feature_names),
            **{f"test_{k}": v for k, v in test_metrics.items()}
        })

        pred_df = meta_test.copy()
        pred_df["dataset"] = dataset
        pred_df["target"] = target
        pred_df["selected_architecture"] = selected_architecture
        pred_df["actual"] = y_test
        pred_df["predicted"] = test_pred
        pred_df["residual_actual_minus_predicted"] = pred_df["actual"] - pred_df["predicted"]
        prediction_frames.append(pred_df)

        prefix = f"{dataset}_{safe_target(target)}_{selected_architecture.lower()}"
        model_path = MODEL_DIR / f"{prefix}.keras"
        imputer_path = MODEL_DIR / f"{prefix}_imputer.joblib"
        scaler_path = MODEL_DIR / f"{prefix}_scaler.joblib"
        feature_path = MODEL_DIR / f"{prefix}_features.json"
        selected["model"].save(model_path)
        joblib.dump(imputer, imputer_path)
        joblib.dump(scaler, scaler_path)
        feature_path.write_text(json.dumps({
            "dataset": dataset,
            "target": target,
            "architecture": selected_architecture,
            "sequence_length": int(X_train.shape[1]),
            "feature_columns_in_order": feature_names,
            "scaling": "StandardScaler fitted on flattened training sequences only",
            "imputation": "Median imputer fitted on flattened training sequences only",
            "random_seed": SEED
        }, indent=2), encoding="utf-8")
        artifact_rows.append({
            "dataset": dataset, "target": target,
            "selected_architecture": selected_architecture,
            "model_path": str(model_path),
            "imputer_path": str(imputer_path),
            "scaler_path": str(scaler_path),
            "feature_manifest_path": str(feature_path)
        })

        for split_name, split_data in split.items():
            _, _, split_meta = split_data
            split_rows.append({
                "dataset": dataset, "target": target, "split": split_name,
                "rows": len(split_meta),
                "target_date_start": str(split_meta["target_match_date"].min().date()),
                "target_date_end": str(split_meta["target_match_date"].max().date())
            })
        task_status_rows.append({
            "dataset": dataset, "target": target, "status": "COMPLETED", "detail": ""
        })
        print(f"COMPLETED: {dataset}/{target}; selected {selected_architecture}")
    except Exception as exc:
        task_status_rows.append({
            "dataset": dataset, "target": target, "status": "FAILED", "detail": repr(exc)
        })
        print(f"FAILED: {dataset}/{target}: {exc}")

metrics_df = pd.DataFrame(all_metrics)
predictions_df = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()
history_df = pd.DataFrame(history_frames)
selection_df = pd.DataFrame(selection_rows)
split_report = pd.DataFrame(split_rows)
artifact_df = pd.DataFrame(artifact_rows)
status_df = pd.DataFrame(task_status_rows)
display(status_df)
display(selection_df)
display(metrics_df)


COMPLETED: batting/next_match_runs; selected LSTM
COMPLETED: bowling/next_match_wickets; selected LSTM
COMPLETED: bowling/next_match_runs_conceded; selected GRU


,dataset,target,status,detail
0,batting,next_match_runs,COMPLETED,
1,bowling,next_match_wickets,COMPLETED,
2,bowling,next_match_runs_conceded,COMPLETED,


,dataset,target,architecture,epochs_run,validation_MAE,validation_RMSE,validation_R2
0,batting,next_match_runs,LSTM,36,16.311644,21.884249,0.106725
1,batting,next_match_runs,GRU,48,16.338200,21.845088,0.109919
2,bowling,next_match_wickets,LSTM,17,0.802349,1.040258,0.026574
3,bowling,next_match_wickets,GRU,27,0.806429,1.039475,0.028038
4,bowling,next_match_runs_conceded,LSTM,40,8.462542,10.676995,0.134718
5,bowling,next_match_runs_conceded,GRU,26,8.458755,10.697968,0.131315


,dataset,target,selected_architecture,train_samples,validation_samples,test_samples,feature_count,test_MAE,test_RMSE,test_R2,test_mean_residual_actual_minus_predicted
0,batting,next_match_runs,LSTM,11074,2373,2374,15,16.700930,22.505660,0.105188,2.578387
1,bowling,next_match_wickets,LSTM,8634,1850,1851,15,0.782648,0.999579,0.019520,0.029028
2,bowling,next_match_runs_conceded,GRU,8634,1850,1851,15,9.509530,11.931938,0.049363,3.587987


## 4. Export reports, artifacts manifest and documentation

Training can take time on a laptop. If a task fails, preserve the error in the status report and resolve it before making performance claims. The selected model is chosen by validation MAE, with validation RMSE as tie-breaker.

In [10]:
REPORT_DIR.mkdir(parents=True, exist_ok=True)
metrics_df.to_csv(REPORT_DIR / "selected_lstm_gru_test_metrics.csv", index=False)
predictions_df.to_csv(REPORT_DIR / "selected_lstm_gru_test_predictions.csv", index=False)
history_df.to_csv(REPORT_DIR / "training_history.csv", index=False)
selection_df.to_csv(REPORT_DIR / "validation_architecture_comparison.csv", index=False)
split_report.to_csv(REPORT_DIR / "chronological_split_summary.csv", index=False)
artifact_df.to_csv(REPORT_DIR / "model_artifacts_manifest.csv", index=False)
status_df.to_csv(REPORT_DIR / "task_status.csv", index=False)

def frame_text(frame):
    return frame.to_string(index=False) if frame is not None and not frame.empty else "No rows produced."

doc = f"""# Day 12 — LSTM/GRU Regression Training

## Task status
```text
{frame_text(status_df)}
```

## Validation architecture comparison
```text
{frame_text(selection_df)}
```

## Selected architecture test metrics
```text
{frame_text(metrics_df)}
```

## Artifacts
```text
{frame_text(artifact_df)}
```

## Method
- Sequence length: 5 historical matches.
- Architectures: LSTM and GRU, one scalar regression output per target.
- Targets: `next_match_runs`, `next_match_wickets`, `next_match_runs_conceded`.
- Split: chronological sample-level 70% train / 15% validation / 15% test by target match date.
- Preprocessing: median imputation and standard scaling fitted on flattened training sequences only.
- Training: Adam optimizer, MSE loss, MAE metric, early stopping on validation loss, `shuffle=False`.
- Selection: lowest validation MAE, validation RMSE as tie-breaker.
- Test partition: evaluated after architecture selection; not used to choose the architecture.
- Seed: {SEED}.
- TensorFlow available at run time: {TENSORFLOW_AVAILABLE}.

## Limitations
- Sequence split is sample-level; samples from different players in the same match may cross split boundaries.
- This experiment uses a small fixed architecture and is a baseline, not a claim of optimal deep-learning performance.
- Validate all feature definitions for leakage and review the actual task status before accepting the metrics.

## Outputs
- `reports/day12/selected_lstm_gru_test_metrics.csv`
- `reports/day12/selected_lstm_gru_test_predictions.csv`
- `reports/day12/training_history.csv`
- `reports/day12/validation_architecture_comparison.csv`
- `reports/day12/chronological_split_summary.csv`
- `reports/day12/model_artifacts_manifest.csv`
- `reports/day12/task_status.csv`
- `models/day12/`
- `docs/day12_lstm_gru_training.md`

## Next
Day 13: error analysis and comparison of the deep-learning models against the Day 8 tree-based baselines.
"""
doc_path = DOCS_DIR / "day12_lstm_gru_training.md"
doc_path.write_text(doc, encoding="utf-8")
print("Saved documentation:", doc_path)
display(status_df)

Saved documentation: D:\Git\Personal\ipl-player-performance\docs\day12_lstm_gru_training.md


,dataset,target,status,detail
0,batting,next_match_runs,COMPLETED,
1,bowling,next_match_wickets,COMPLETED,
2,bowling,next_match_runs_conceded,COMPLETED,
